## Open In Colab\n\n[![Run in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/achyutmorang/waymax-simulation-experiments/blob/main/experiments/risk-uq-suite/notebooks/miscalibration_probe_colab.ipynb)


# Risk/UQ Paper Track: Miscalibration Probe

## Objective
Demonstrate, before risk-model training, whether planner-side confidence proxies are miscalibrated and whether budgeted decisions can be over/under-confident.

## Inputs
- Existing risk dataset artifact if available (`*_risk_dataset.parquet`)
- Otherwise one lightweight simulator context for probe dataset construction

## Outputs
- Probe summary/per-shift calibration metrics
- Reliability bins and selective-risk curves
- Threshold diagnostics for budgeted decisions

## Success Criteria
- At least one planner proxy variant shows non-trivial calibration gap (ECE/NLL gap) on nominal and/or shift subsets.
- Threshold diagnostics expose whether `risk_control_fail_budget` can be false-safe or overly conservative.


## Hypotheses Being Tested
- **H1:** Planner-side proxy risks are not perfectly calibrated on `nominal_clean`.
- **H2:** Calibration gaps are larger on shifted/high-interaction subsets than on nominal.
- **H3:** At budget threshold `tau = risk_control_fail_budget`, empirical failure among accepted samples can exceed `tau` (false-safe risk).

## Methodology
1. Build/load candidate-level dataset rows with closed-loop rollout outcomes.
2. Construct planner-only risk proxies from distribution trace features:
   - `planner_risk_top1_proxy`
   - `planner_risk_entropy_proxy`
   - `planner_risk_combo_proxy`
3. Benchmark these proxies against realized failure labels using ECE/NLL/Brier/reliability tables.
4. Run threshold diagnostics to detect over/under-confidence under budgeted acceptance.


## Step 1 - Deterministic Bootstrap Constants


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/achyutmorang/waymax-simulation-experiments.git'
REPO_DIR = '/content/waymax-simulation-experiments'
REPO_BRANCH = 'main'
EXPERIMENT_SLUG = 'risk-uq-suite'
EXPERIMENT_CONFIG_PATH = f'{REPO_DIR}/configs/experiments/{EXPERIMENT_SLUG}.json'
REQUIRED_DRIVE_FOLDER = '/content/drive/MyDrive/waymax_experiments'

runtime_cfg_overrides = {
    'verify_drive_access_every_run': False,
    'force_reinstall': False,
    'auto_restart_after_setup': True,
    'strict_lockfile_check': True,
    'setup_cache_enabled': True,
    'revalidate_core_imports_on_cache_hit': True,
    'setup_cache_path': '/content/.closedloop_setup_cache.json',
    'force_module_hot_reload': True,
}

content_root = Path('/content')
content_root.mkdir(parents=True, exist_ok=True)
try:
    _ = os.getcwd()
except FileNotFoundError:
    os.chdir(str(content_root))


## Step 2 - Storage Setup


In [ ]:
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except Exception as exc:
    print('[storage] google.colab drive mount unavailable:', exc)

drive_root = Path('/content/drive/MyDrive')
if not drive_root.exists():
    raise RuntimeError('Drive root not found at /content/drive/MyDrive. Ensure Google Drive is mounted.')

required_root = Path(REQUIRED_DRIVE_FOLDER)
required_root.mkdir(parents=True, exist_ok=True)
probe = required_root / '.risk_uq_storage_probe'
probe.write_text('ok')
probe.unlink(missing_ok=True)
print('[storage] ready:', required_root)


## Step 3 - Repo Sync + Runtime Bootstrap


In [ ]:
if not Path(REPO_DIR).exists():
    subprocess.run(['git', 'clone', '--depth', '1', '-b', REPO_BRANCH, REPO_URL, REPO_DIR], check=True)

for path in (REPO_DIR, f'{REPO_DIR}/src'):
    if path not in sys.path:
        sys.path.insert(0, path)

from src.platform import ColabRuntimeConfig, bootstrap_colab_runtime_with_config

runtime_cfg = ColabRuntimeConfig(
    repo_url=REPO_URL,
    repo_dir=REPO_DIR,
    repo_branch=REPO_BRANCH,
    required_drive_folder=REQUIRED_DRIVE_FOLDER,
    **runtime_cfg_overrides,
)
bootstrap = bootstrap_colab_runtime_with_config(runtime_cfg)

globals()['_RISK_UQ_REPO_REV'] = str(bootstrap.repo_sync.repo_rev)
setup_result = bootstrap.setup.result

print('Working directory:', os.getcwd())
print('Repo commit:', bootstrap.repo_sync.repo_rev)
print('Drive mounted:', bootstrap.drive_status.mounted)
print('[setup] result:', setup_result)
if bool(setup_result.get('restart_required', False)) and (not bool(runtime_cfg.auto_restart_after_setup)):
    print('[setup] restart_required=True; restart runtime before continuing.')


## Step 4 - Configuration + Run Context


In [ ]:
from src.workflows import initialize_run_context, load_experiment_config, report_run_context

EXPERIMENT_CFG = load_experiment_config(
    slug=EXPERIMENT_SLUG,
    repo_root=REPO_DIR,
    default_on_missing=False,
)
run_cfg = dict(EXPERIMENT_CFG.get('run', {}))

# Mandatory contract fields
RUN_NAME = str(run_cfg.get('run_name', '')).strip()
RUN_PREFIX = str(run_cfg.get('run_prefix', 'risk_uq')).strip() or 'risk_uq'
PERSIST_ROOT = str(run_cfg.get('persist_root', '/content/drive/MyDrive/waymax_experiments/risk_uq_runs/v1')).strip()
N_SHARDS = int(max(1, int(run_cfg.get('n_shards', 1))))
SHARD_ID = run_cfg.get('shard_id', 'auto')
RESUME_FROM_EXISTING = bool(run_cfg.get('resume_from_existing', True))
RUN_ENABLED = bool(run_cfg.get('run_enabled', True))

RUN_TAG_PREFIX = str(run_cfg.get('run_tag_prefix', RUN_PREFIX)).strip() or RUN_PREFIX
RUN_MODE_CFG = str(run_cfg.get('run_mode', 'auto')).strip().lower() or 'auto'
RUN_MODE = RUN_MODE_CFG if RESUME_FROM_EXISTING else 'fresh'
RUN_TAG = RUN_NAME

AUTO_RUN_MAIN_LOOP_WHEN_READY = False
RUN_MAIN_LOOP_OVERRIDE = False
WARN_ON_CONFIG_DRIFT = True

run_context = initialize_run_context(
    run_tag=RUN_TAG,
    persist_root=PERSIST_ROOT,
    n_shards=N_SHARDS,
    shard_id=SHARD_ID,
    auto_run_main_loop_when_ready=bool(AUTO_RUN_MAIN_LOOP_WHEN_READY),
    run_main_loop_override=RUN_MAIN_LOOP_OVERRIDE,
    run_tag_prefix=RUN_TAG_PREFIX,
    planner_backend='latentdriver',
    planner_surprise_name='predictive_seq_w2',
    auto_generate_run_tag_if_empty=True,
    resume_mode=RUN_MODE,
    warn_on_config_drift=bool(WARN_ON_CONFIG_DRIFT),
)

cfg = run_context.cfg
search_cfg = run_context.search_cfg
RUN_TAG = str(run_context.run_tag)
RUN_NAME = str(RUN_NAME or RUN_TAG)
SHARD_ID = int(run_context.shard_id)
run_prefix = run_context.run_prefix

print('EXPERIMENT_CONFIG_PATH =', EXPERIMENT_CONFIG_PATH)
print('run_prefix (flat artifacts) =', run_prefix)
print('RUN_PREFIX/RUN_NAME (contract dir) =', RUN_PREFIX, RUN_NAME)
print('RUN_ENABLED =', RUN_ENABLED, ' RESUME_FROM_EXISTING =', RESUME_FROM_EXISTING)
report_run_context(run_context, display_fn=display)

# Probe knobs
cfg.uq_eval_probability_bins = 15
cfg.risk_control_fail_budget = 0.20
cfg.uq_shift_suites = ('nominal_clean', 'hist_prim_shift', 'fut_prim_shift', 'hist_rmv_shift', 'high_interaction_holdout')
cfg


## Step 5 - Fast-Fail Validation


In [ ]:
import numpy as np
import pandas as pd
from src.workflows import (
    build_full_simulation_context,
    load_existing_risk_dataset_artifact,
    run_risk_training_notebook_gates,
)

existing_dataset_df = load_existing_risk_dataset_artifact(cfg.run_prefix)
if RUN_MODE == 'fresh':
    existing_dataset_df = pd.DataFrame()

needs_simulation_context = bool(existing_dataset_df.empty)
runner = None
eval_idx = None

print('existing_dataset_rows =', len(existing_dataset_df))
print('needs_simulation_context =', needs_simulation_context)

if needs_simulation_context:
    sim_bundle = build_full_simulation_context(cfg=cfg, n_shards=N_SHARDS, shard_id=SHARD_ID)
    runner = sim_bundle.runner
    eval_idx = sim_bundle.eval_idx

    gate_bundle = run_risk_training_notebook_gates(
        runner=runner,
        cfg=cfg,
        eval_idx=eval_idx,
        probe_shift_suite='nominal_clean',
    )
    risk_probe_pass = bool(gate_bundle.get('risk_probe_pass', False))
    preflight_pass = bool(gate_bundle.get('preflight_pass', False))
    overall_pass = bool(gate_bundle.get('overall_pass', False))

    display(gate_bundle.get('risk_probe_summary_df', pd.DataFrame()))
    display(gate_bundle.get('preflight_df', pd.DataFrame()))
else:
    gate_bundle = {'failure_reasons': []}
    required_cols = [
        'scenario_id',
        'dist_top1_weight',
        'dist_entropy',
        'dist_num_components',
        'failure_proxy_h15',
    ]
    required_ok = all(col in existing_dataset_df.columns for col in required_cols)

    finite_ok = False
    if required_ok and len(existing_dataset_df) > 0:
        sample = existing_dataset_df.loc[:, ['dist_top1_weight', 'dist_entropy', 'dist_num_components', 'failure_proxy_h15']].head(4096)
        finite_ok = bool(np.isfinite(sample.to_numpy(dtype=float)).all())

    risk_probe_pass = bool((len(existing_dataset_df) > 0) and required_ok and finite_ok)
    preflight_pass = bool(risk_probe_pass)
    overall_pass = bool(risk_probe_pass and preflight_pass)

print('risk_probe_pass =', risk_probe_pass)
print('preflight_pass =', preflight_pass)
print('overall_pass =', overall_pass)

if not overall_pass:
    raise RuntimeError(f"Miscalibration probe gates failed: {gate_bundle.get('failure_reasons', [])}")


## Step 6 - Manifest + Contract Layout (Gates Passed)


In [ ]:
from src.workflows import write_contract_storage_mirror, write_notebook_contract_manifest

manifest_path = write_notebook_contract_manifest(
    run_prefix=cfg.run_prefix,
    run_tag=RUN_TAG,
    cfg=cfg,
    search_cfg=search_cfg,
    n_shards=N_SHARDS,
    shard_id=SHARD_ID,
    notebook_name='miscalibration_probe_colab',
    stage='miscalibration_probe_started',
    repo_dir=REPO_DIR,
    git_commit=globals().get('_RISK_UQ_REPO_REV', ''),
    quick_probe_pass=bool(risk_probe_pass),
    preflight_pass=bool(preflight_pass),
)
contract_paths = write_contract_storage_mirror(
    persist_root=PERSIST_ROOT,
    run_prefix=RUN_PREFIX,
    run_name=RUN_NAME,
    run_prefix_path=cfg.run_prefix,
    cfg=cfg,
    search_cfg=search_cfg,
    n_shards=N_SHARDS,
    shard_id=SHARD_ID,
    stage='miscalibration_probe_started',
    git_commit=str(globals().get('_RISK_UQ_REPO_REV', '')),
    resume_from_existing=bool(RESUME_FROM_EXISTING),
    run_enabled=bool(RUN_ENABLED),
)
print('manifest_path =', manifest_path)
print('contract_run_dir =', contract_paths['contract_run_dir'])


## Step 7 - Main Execution (Resume-Aware Probe)


In [ ]:
from src.workflows import run_miscalibration_probe_flow

probe_bundle = None
if not bool(RUN_ENABLED):
    print('[main] skipped: RUN_ENABLED=False')
else:
    probe_bundle = run_miscalibration_probe_flow(
        cfg=cfg,
        dataset_df=existing_dataset_df if not existing_dataset_df.empty else None,
        runner=runner,
        eval_idx=eval_idx,
        run_prefix=cfg.run_prefix,
        resume_mode=RUN_MODE,
    )
    print('loaded_from_existing =', probe_bundle.loaded_from_existing)
    display(probe_bundle.benchmark_bundle.summary_df)


## Step 8 - Evaluation/Diagnostics


In [ ]:
if probe_bundle is None:
    print('[report] no run output because RUN_ENABLED=False')
else:
    display(probe_bundle.benchmark_bundle.per_shift_df.head(30))
    display(probe_bundle.benchmark_bundle.reliability_df.head(30))
    display(probe_bundle.threshold_df.head(30))
    if hasattr(probe_bundle, 'proxy_calibration_df'):
        display(probe_bundle.proxy_calibration_df.head(20))
    if hasattr(probe_bundle, 'leakage_df'):
        display(probe_bundle.leakage_df.head(20))
    if hasattr(probe_bundle, 'shift_profile_df'):
        display(probe_bundle.shift_profile_df.head(30))
    if hasattr(probe_bundle, 'class_balance_df'):
        display(probe_bundle.class_balance_df.head(30))

    key = probe_bundle.threshold_df.copy()
    if (not key.empty) and ('variant' in key.columns):
        cols = ['shift_suite', 'variant', 'empirical_failure_given_accepted', 'threshold', 'budget_violated']
        cols = [c for c in cols if c in key.columns]
        print('[diagnostic] budget check by shift/variant')
        display(key.loc[:, cols].head(30))


## Step 9 - Export + Manifest (Completed)


In [ ]:
if probe_bundle is None:
    stage_name = 'miscalibration_probe_skipped'
    artifact_paths = {}
    metrics_path = None
    extra = {'run_skipped': 1}
else:
    stage_name = 'miscalibration_probe_completed'
    artifact_paths = dict(probe_bundle.artifact_paths)
    metrics_path = str(artifact_paths.get('miscalibration_probe_summary', '')) or None
    extra = {
        'loaded_from_existing': int(bool(probe_bundle.loaded_from_existing)),
        'summary_rows': int(len(probe_bundle.benchmark_bundle.summary_df)),
        'per_shift_rows': int(len(probe_bundle.benchmark_bundle.per_shift_df)),
        'threshold_rows': int(len(probe_bundle.threshold_df)),
    }

manifest_path = write_notebook_contract_manifest(
    run_prefix=cfg.run_prefix,
    run_tag=RUN_TAG,
    cfg=cfg,
    search_cfg=search_cfg,
    n_shards=N_SHARDS,
    shard_id=SHARD_ID,
    notebook_name='miscalibration_probe_colab',
    stage=stage_name,
    repo_dir=REPO_DIR,
    git_commit=globals().get('_RISK_UQ_REPO_REV', ''),
    extra_fields=extra,
)

contract_paths = write_contract_storage_mirror(
    persist_root=PERSIST_ROOT,
    run_prefix=RUN_PREFIX,
    run_name=RUN_NAME,
    run_prefix_path=cfg.run_prefix,
    cfg=cfg,
    search_cfg=search_cfg,
    n_shards=N_SHARDS,
    shard_id=SHARD_ID,
    stage=stage_name,
    git_commit=str(globals().get('_RISK_UQ_REPO_REV', '')),
    resume_from_existing=bool(RESUME_FROM_EXISTING),
    run_enabled=bool(RUN_ENABLED),
    artifact_paths=artifact_paths,
    metrics_csv_path=metrics_path,
    extra_fields=extra,
)
print('manifest_path =', manifest_path)
print('contract_run_manifest =', contract_paths['contract_run_manifest'])
